# B2-023-generative-models-diffusion — Practice p22 — Solution

**Type:** scenario · **Difficulty:** core · **Concepts:** generative-adversarial-network

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Solution

Coverage counts are for $n=512$ samples and $K=4$ components, so the p08 threshold is $\lceil512/32\rceil=16$.

### (a) Diagnoses

**Covered-mode counts for every row** (computed in the cell below):

| run | steps | counts | covered | inside any radius |
|---|---|---|---:|---:|
| A | 100–149 | (0, 0, 171, 19) | 2 | 190/512 |
| A | 200–249 | (0, 3, 2, 207) | 1 | 212/512 |
| A | 300–349 | (0, 0, 9, 188) | 1 | 197/512 |
| B | 100–149 | (0, 0, 244, 21) | 2 | 265/512 |
| B | 200–249 | (0, 0, 251, 30) | 2 | 281/512 |
| B | 300–349 | (0, 0, 255, 34) | 2 | 289/512 |

**Run A — mode collapse.**
*$\mathcal L_D$ column:* window means $1.04$–$1.30$, near $2\log2=1.386$: the discriminator is not winning.
*$\mathcal L_G$ column:* moderate ($0.71$–$1.23$), not growing.
*Mean $D$ columns:* $D(\text{train})\approx0.55$ and $D(\text{fake})\approx0.41$–$0.49$: the discriminator barely separates real from fake.
*Coverage column:* nearly all counted samples sit on one blob (171 on component 2, then 207 and 188 on component 3), with only 1–2 components covered.
A balanced game with samples concentrated on one blob is mode collapse.

**Run B — discriminator overpowering.**
*$\mathcal L_D$ column:* falls to $0.007$–$0.23$, far below $2\log2$.
*$\mathcal L_G$ column:* large ($3.6$, then $8.3$, $8.1$, $5.7$), because fake logits are far below zero.
*Mean $D$ columns:* $D(\text{train})$ up to $0.997$ and $D(\text{fake})$ down to $0.0003$: the discriminator separates the sets almost perfectly.
*Coverage column:* only two components, and $223$–$247$ of $512$ samples lie outside every radius (stranded).
The discriminator dominates and the generator receives gradients only from the steep side of the sigmoid: overpowering.

### (b) Why Run A's $\mathcal L_D$ is not evidence of a good generator

$\mathcal L_D\approx2\log2$ says only that the **current** discriminator cannot separate real from generated rows; it is reached by a perfect generator, but equally by a discriminator that is weak or has just been fooled by a generator concentrated on one region it currently accepts.
The coverage counts show exactly that: the mass **hops** from component 2 (steps 100–149) to component 3 (steps 200–349).
Mode hopping means the game is cycling rather than converging: the generator chases whichever blob the discriminator currently undervalues, the discriminator learns to reject it, and the generator moves on, so the loss stays balanced while coverage never improves.

### (c) Remedies

**Run A (mode collapse): choose (v)** — lower the generator's learning rate relative to the discriminator's and track coverage counts over training.
A slower generator gives the discriminator time to learn the whole data distribution, so it penalizes the occupied blob before the generator jumps wholesale to another, damping the hopping; tracking coverage counts is the only signal that the cure works, since the loss already looked fine.
*Rejected for A: (iii) the saturating loss* — it changes nothing about the hopping dynamics and risks vanishing generator gradients whenever the discriminator does gain the upper hand, stalling training. (Also rejected: (i), multiplying the discriminator's rate by 10, risks overshooting into Run B's overpowering.)

**Run B (overpowering): choose (ii)** — lower the discriminator's learning rate to the generator's and re-check coverage.
This restores balance at the source (the discriminator learns too fast relative to the generator), letting $\mathcal L_D$ rise toward $2\log2$ so the generator receives useful gradient; coverage must be re-checked because a balanced loss alone could still hide collapse (as Run A shows).
*Rejected for B: (iv) five discriminator steps per generator step* — it strengthens the already dominant discriminator and drives $\mathcal L_G$ even higher. (Option (iii) would also fail: with $\ell_{\text{fake}}\ll0$ the saturating gradient $-\sigma(\ell)\approx0$, so the generator would learn even less; option (i) makes the imbalance ten times worse.)

### (d) Certificates to re-run

After each remedy, re-run p17's certificates with their exact thresholds:

1. **(B) Discriminator band:** the mean of the last 50 discriminator losses lies within $0.95$ of $2\log2$, i.e. in $[0.4363,\,2.3363]$.
2. **(D) Coverage:** `covered >= 2` under the p08 rule with radius $0.6$ on 512 fixed-noise samples and threshold $\lceil512/32\rceil=16$.
3. **(E) No memorization signal:** `abs(p_train - p_held) <= 0.10` for the mean discriminator probabilities on training and held-out rows.

(Alongside: (C) every generator loss finite and at most $5.0$, which Run B as logged would fail.)
The learning rates are hyperparameters, so they must be chosen using training data only (training traces and training-side coverage); if held-out rows influenced the choice, the held-out comparison in (E) would no longer be an independent check — the held-out set would have been fitted through the selection, and its certificate would be optimistically biased (selection leakage).

In [ ]:
import math

import torch

need = math.ceil(512 / (8 * 4))
rows = {
    ("A", "100-149"): (0, 0, 171, 19),
    ("A", "200-249"): (0, 3, 2, 207),
    ("A", "300-349"): (0, 0, 9, 188),
    ("B", "100-149"): (0, 0, 244, 21),
    ("B", "200-249"): (0, 0, 251, 30),
    ("B", "300-349"): (0, 0, 255, 34),
}
covered = {key: sum(c >= need for c in counts) for key, counts in rows.items()}
inside = {key: sum(counts) for key, counts in rows.items()}
two_log_two = 2 * math.log(2)
band = (two_log_two - 0.95, two_log_two + 0.95)
run_a_ld = [1.281, 1.041, 1.297, 1.112]
run_b_ld = [0.348, 0.007, 0.049, 0.231]
run_b_lg = [3.598, 8.302, 8.087, 5.702]
# Run B, steps 100-149: L_G = mean(-log D(fake)) >= -log(mean D(fake)) by Jensen, so mean D(fake) = 0.0003 forces L_G >= 8.11
lg_from_dfake = -math.log(0.0003)
print("threshold", need)
for key in rows:
    print(key, rows[key], "covered", covered[key], "inside", inside[key])
print("band", [round(v, 4) for v in band], " -log(0.0003) =", round(lg_from_dfake, 3))

### Answer check

In [ ]:
assert need == 16
assert [covered[k] for k in rows] == [2, 1, 1, 2, 2, 2]
assert [inside[k] for k in rows] == [190, 212, 197, 265, 281, 289]
assert all(abs(v - two_log_two) <= 0.36 for v in run_a_ld)          # Run A balanced near 2 log 2
assert all(v < 0.4 for v in run_b_ld) and max(run_b_lg) > 5.0      # Run B: D wins, G loss large
assert round(band[0], 4) == 0.4363 and round(band[1], 4) == 2.3363
assert not all(lo <= v <= hi for v in run_b_ld for lo, hi in [band])  # Run B's late windows fall below the band
assert lg_from_dfake > 8.0 and run_b_lg[1] >= lg_from_dfake